# Per-gene TWAS Z + Mendelian Randomization

## Description

Fan out per row of a per-gene manifest. Each row pairs a gene's `TwasWeights` RDS with the matching `GwasSumStats` RDS for that gene's home LD block; the SoS step runs `twas.R` per row, which calls `pecotmr::causalInferencePipeline()` to compute the gene's TWAS Z + (optional) MR statistics. Optional per-gene `FineMappingResult` is wired through too.

## Inputs

- `--manifest` &mdash; per-gene manifest TSV with columns:
    - `gene_id` &mdash; gene identifier (unique; used in the output filename).
    - `twas_weights_rds` &mdash; path to the per-gene `TwasWeights` RDS (single TwasWeightsEntry).
    - `gwas_sumstats_rds` &mdash; path to the per-LD-block `GwasSumStats` RDS that covers the gene's home block.
    - `fine_mapping_result_rds` &mdash; optional; path to the matching per-gene `FineMappingResult` RDS (leave blank to skip).
- `--study` &mdash; study identifier (used in output filenames).
- `--mr-pip-cutoff` &mdash; pass-through (default 0.5).
- `--mr-method` &mdash; pass-through (`"ivwPerVariant"` or `"csAware"`; default `"ivwPerVariant"`).

## Output

- `{cwd}/{study}.{gene_id}.twas.rds` per gene.


## Example

```bash
sos run pipeline/twas.ipynb twas \
    --cwd output --modular-script-dir /path/to/code/script \
    --study protocol_example_chr22 \
    --manifest /tmp/gwas_smoke/twas_manifest.tsv
```

Manifest example:
```
gene_id	twas_weights_rds	gwas_sumstats_rds	fine_mapping_result_rds
ENSG00000130538	/tmp/gwas_smoke/blessed_tw.rds	/tmp/gwas_smoke/blocks/chr22_10516173_17414263.rds	
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: manifest = path
parameter: mr_pip_cutoff = 0.5
parameter: mr_method = 'ivwPerVariant'
parameter: modular_script_dir = path('code/script')
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[twas]
# Read the per-gene manifest. Header may be canonical (`gene_id`,
# `twas_weights_rds`, `gwas_sumstats_rds`, `fine_mapping_result_rds`) and
# may carry a leading `#`. fine_mapping_result_rds is optional per-row
# (empty cell = skip).
def _read_manifest(path):
    rows = []
    with open(path) as f:
        hdr = next(f).rstrip('\n').lstrip('#').split('\t')
        required = {'gene_id', 'twas_weights_rds', 'gwas_sumstats_rds'}
        missing = required - set(hdr)
        if missing:
            raise ValueError(
                f"Manifest missing required column(s): {sorted(missing)}")
        for line in f:
            parts = line.rstrip('\n').split('\t')
            row = dict(zip(hdr, parts))
            if not row.get('gene_id'):
                continue
            rows.append(row)
    if not rows:
        raise ValueError(f"Manifest {path} is empty.")
    ids = [r['gene_id'] for r in rows]
    if len(set(ids)) != len(ids):
        raise ValueError(
            "Manifest has duplicate gene_id values; each row must be unique.")
    return rows
rows = _read_manifest(manifest)
gene_ids   = [r['gene_id'] for r in rows]
tw_paths   = [r['twas_weights_rds'] for r in rows]
gss_paths  = [r['gwas_sumstats_rds'] for r in rows]
fmr_paths  = [r.get('fine_mapping_result_rds', '') or '' for r in rows]
input: for_each = ['gene_ids', 'tw_paths', 'gss_paths', 'fmr_paths']
fmr_arg = f"--fine-mapping-result {_fmr_paths}" if _fmr_paths else ""
output: f"{cwd}/{study}.{_gene_ids}.twas.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/twas.R \
        --twas-weights ${_tw_paths} \
        --gwas-sumstats ${_gss_paths} \
        ${fmr_arg} \
        --mr-pip-cutoff ${mr_pip_cutoff} \
        --mr-method ${mr_method} \
        --output ${_output}
